# 23. The control never run, and the loss never tried under K

Two branches, one per card, in a single session of roughly three to five
hours. Nothing in this notebook needs editing: run it as it is.

Two branches that are not ideas. One is a control this project should
have run first, the other is prior art it has been missing.

**BM** is one ResNet-18 at width 1.00 and nothing else. All sixty five
configs here train every width, so the table cannot say what multi-width
training costs the widest width - and that number gates a whole family of
work. A's width 1.00 reads 75.30. If BM lands well above it, the hard
one-hot label at the widest is being drowned by the narrow widths' much
larger losses, and the fix belongs at the label. If BM lands near 75.30,
that family is closed. The literature disagrees with itself on the sign:
the US-Net paper has MobileNet v1 *better* multi-width than solo by 0.9
and v2 worse by 0.3, while SlimCLR's supervised control is 76.6 solo
against 76.0 slimmable. A reviewer asks for this number either way.

BM also tests all sixteen widths afterwards, so the row shows what a
model trained at one width does when it is sliced anyway - the other
thing nothing here has measured.

**BK** is K with AlphaNet's alpha-divergence in place of the soft cross
entropy. Plain KL is zero-avoiding, so a student made to match a teacher
it cannot represent over-estimates the teacher's uncertainty; the
alpha-divergence penalises over- and under-estimation both, and the paper
applies it to slimmable networks over exactly this width range. The loss
has been in loss_ops since branch B and has never been put underneath K.

B is that loss on A and came out at 73.08, below A - but B has no
transport term, and every divergence in this project has depended on
which tier it sits at: KL, Jeffreys and Wasserstein all fail at the logit
tier while Wasserstein at the feature tier is worth +0.75. So B does not
settle what alpha does under K. Prior art either way: if it helps, the
baseline should have had it and the comparison has to be redrawn against
K plus alpha.

## `bm_solo_full`

One ResNet-18 at width 1.00 and nothing else. The control this project has never run. All 65 configs train every width, so the table cannot say what the multi-width training costs the widest width - and that number gates a whole family of ideas. If 1.00 here lands well above A's 75.30, the hard label at the widest is being drowned by the narrow widths' much larger losses and the fix belongs at the label. If it lands near 75.30, that family is closed. The literature disagrees with itself on the size: the US-Net paper has MobileNet v1 better multi-width than solo by 0.9 and MobileNet v2 worse by 0.3, while SlimCLR's supervised control is 76.6 solo against 76.0 slimmable. A reviewer asks for this number whichever way it goes. narrow_start_epoch past the end leaves training_widths returning [high] alone every epoch: one forward pass at width 1.00, no narrow widths, no distillation. Same path the warm-up branches use, pushed to the limit, so nothing here is a new code path. The sixteen widths are still tested afterwards, so this row also shows what a model trained at one width does when it is sliced anyway - the other thing nothing in this table has measured.

## `bk_alpha_on_k`

K with the alpha-divergence in place of the soft cross entropy. AlphaNet's argument is that plain KL is zero-avoiding, so a student made to match a teacher it cannot represent over-estimates the teacher's uncertainty. The alpha-divergence penalises over- and under-estimation both, and the paper applies it to slimmable networks over exactly this width range. B is that loss on A and came out at 73.08, below A. But B has no transport term, and every reading of a divergence in this project has depended on which tier it sits at: KL, Jeffreys and Wasserstein all fail at the logit tier while Wasserstein at the feature tier is worth +0.75. So B does not settle what alpha does underneath K. Prior art either way. If it helps, the baseline K should have had it and the comparison in the paper has to be redrawn against K+alpha.


## Before you start

* Accelerator **GPU T4 x2**, Internet **On**
* Add the CIFAR-100 dataset as an input
* **Save Version -> Save & Run All (Commit)**, not the interactive run.
  An interactive session ends when the browser closes.

The checks at the top run on the CPU and stop the session in about two
minutes if anything is wrong, before a card is touched. Note that the
branch suite mimics the training loop rather than running it, so for
these branches the smoke step below is what actually exercises the new
code. Do not skip it.

If the session times out partway, attach its output to a new copy and
name the logs directory in `RESUME_FROM`. Every epoch writes a
checkpoint, so at most one is lost.

## When it finishes

Send back the final table. Pasting the output of the last cell is enough.

In [ ]:
# Fixed for this notebook. Notebook 23 of 24.
BRANCHES = ['bm_solo_full', 'bk_alpha_on_k']

SMOKE_FIRST = True

REPO_URL = 'https://github.com/duyh80456-code/new-pruning.git'
REPO_BRANCH = 'nhan'

CIFAR_DIR = ('/kaggle/input/datasets/nlnk1607/cifar100/cifar-100-python')

# To carry a timed-out session forward, attach its output and name the
# logs directory. Leave empty to start fresh.
RESUME_FROM = ''

In [ ]:
import os
import queue
import re
import shutil
import subprocess
import sys
import threading
import time

import torch

n_gpu = torch.cuda.device_count()
print('torch', torch.__version__, '| gpus', n_gpu)
for i in range(n_gpu):
    print('  {}: {}'.format(i, torch.cuda.get_device_properties(i).name))

WORK = '/kaggle/working'
CODE = os.path.join(WORK, 'new-pruning')
if not os.path.isdir(CODE):
    subprocess.run(
        ['git', 'clone', '-b', REPO_BRANCH, REPO_URL, CODE], check=True)
os.chdir(CODE)

available = sorted(
    name[len('cifar100_'):-len('.yml')]
    for name in os.listdir('apps')
    if name.startswith('cifar100_') and name.endswith('.yml'))
print('\nbranches in apps/:')
for name in available:
    print('   ', name)

missing = [b for b in BRANCHES if b not in available]
if missing:
    raise SystemExit('no config for {}'.format(missing))
if n_gpu < len(BRANCHES):
    print('\n{} branches, {} gpu(s): they will run in sequence.'.format(
        len(BRANCHES), n_gpu))

In [ ]:
# What the chosen branches actually differ in, read off the configs
# rather than from the table above, which can drift.
AXES = ('kd_loss', 'cost_source', 'feature_kd', 'feature_align',
        'feature_layers', 'feature_weight', 'tier_weights',
        'horizontal_kd', 'horizontal_where', 'horizontal_loss',
        'horizontal_weight', 'weight_schedule')

settings = {}
for branch in BRANCHES:
    found = {}
    with open('apps/cifar100_{}.yml'.format(branch)) as handle:
        for line in handle:
            key = line.split(':')[0].strip()
            if key in AXES:
                found[key] = line.split(':', 1)[1].strip()
    settings[branch] = found

print('{:20}'.format('') + ''.join(
    '{:>26}'.format(b) for b in BRANCHES))
for axis in AXES:
    values = [settings[b].get(axis, '-') for b in BRANCHES]
    if all(v == '-' for v in values):
        continue
    print('{:20}'.format(axis) + ''.join(
        '{:>26}'.format(v) for v in values))

In [ ]:
TARGET = 'data/cifar-100-python'
if not os.path.isdir(TARGET):
    source = CIFAR_DIR if os.path.isdir(CIFAR_DIR) else None
    if source is None:
        for root, dirs, _ in os.walk('/kaggle/input'):
            if 'cifar-100-python' in dirs:
                source = os.path.join(root, 'cifar-100-python')
                break
    os.makedirs('data', exist_ok=True)
    if source:
        os.symlink(source, TARGET)
        print('linked', source)
    else:
        from torchvision import datasets
        datasets.CIFAR100(root='data', train=True, download=True)
        datasets.CIFAR100(root='data', train=False, download=True)
print(sorted(os.listdir(TARGET)))

## Checks, before a card is touched

Seconds on the CPU. Between them these suites have caught a Sinkhorn solved too loosely to have a correct gradient, an alpha-divergence that destroyed the weights in three steps, a calibration that reset the batch norm statistics and never refilled them, and a feature cost normalized so that its own gradient vanished. Every one of those was silent.

The last suite builds each branch in this notebook and runs two training steps of the real loop, profiling included. Four feature branches once reached Kaggle, passed every loss check, and died in the profiler on the first forward, because nothing local had ever called it.

In [ ]:
# Driven by BRANCHES, not by a list written beside it. The branch suite
# used to name its two branches literally, which quietly checked the wrong
# pair for anyone who changed BRANCHES and nothing else.
suites = ['tests/test_loss_ops.py', 'tests/test_bn_calibration.py',
          'tests/test_kd_variants.py',
          'tests/test_channel_reorder.py']
for suite in suites:
    done = subprocess.run([sys.executable, suite], capture_output=True,
                          text=True)
    print('{:34} {}'.format(
        suite, (done.stdout.strip().splitlines() or ['no output'])[-1]))
    if done.returncode != 0:
        print(done.stdout[-3000:], done.stderr[-2000:])
        raise SystemExit('{} failed'.format(suite))

done = subprocess.run(
    [sys.executable, 'tests/test_all_branches.py'] + BRANCHES,
    capture_output=True, text=True)
print()
print(done.stdout.strip()[-2000:])
if done.returncode != 0:
    print(done.stderr[-2000:])
    raise SystemExit('a branch in {} does not build'.format(BRANCHES))

In [ ]:
if RESUME_FROM:
    os.makedirs('logs', exist_ok=True)
    for name in os.listdir(RESUME_FROM):
        src = os.path.join(RESUME_FROM, name)
        if os.path.isdir(src):
            shutil.copytree(src, os.path.join('logs', name),
                            dirs_exist_ok=True)
            print('restored', name)
else:
    print('starting from scratch')

In [ ]:
VAL_LINE = re.compile(
    r'val\s+([0-9.]+)\s+-1/\d+:\s+loss:\s+([0-9.eE+-]+),\s+'
    r'top1_error:\s+([0-9.]+)')
results = {}


def run_pinned(jobs, quiet=True):
    """one job per card, output interleaved and tagged"""
    lines = queue.Queue()
    procs = {}
    failing = set()

    def pump(label, proc):
        for line in proc.stdout:
            lines.put((label, line.rstrip('\n')))
        proc.wait()
        lines.put((label, None))

    for index, (label, config) in enumerate(jobs):
        env = dict(os.environ)
        env['CUDA_VISIBLE_DEVICES'] = str(index % max(n_gpu, 1))
        proc = subprocess.Popen(
            [sys.executable, '-u', 'train.py', 'app:' + config],
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, env=env)
        procs[label] = proc
        threading.Thread(target=pump, args=(label, proc),
                         daemon=True).start()
        print('[{}] started on gpu {} with {}'.format(
            label, env['CUDA_VISIBLE_DEVICES'], config), flush=True)

    started = time.time()
    remaining = len(jobs)
    while remaining:
        label, line = lines.get()
        if line is None:
            remaining -= 1
            print('[{}] exit code {} after {:.0f} min'.format(
                label, procs[label].returncode,
                (time.time() - started) / 60), flush=True)
            continue
        found = VAL_LINE.search(line)
        if found:
            width, loss, top1 = found.groups()
            results.setdefault(label, {})[float(width)] = (
                float(loss), float(top1))
        # Once a branch starts printing a traceback, stop filtering
        # it. The body is indented, so the rule below would keep the
        # exception and throw away where it came from.
        if 'Traceback (most recent call last)' in line:
            failing.add(label)
        if quiet and label not in failing and (
                line.startswith(('  ', ')', 'Model(', 'Total', 'Item'))
                or not line.strip()):
            continue
        print('[{}] {}'.format(label, line), flush=True)

    return {label: proc.returncode for label, proc in procs.items()}

In [ ]:
if SMOKE_FIRST:
    codes = run_pinned(
        [(b, 'apps/smoke_{}.yml'.format(b)) for b in BRANCHES])
    failed = [b for b, code in codes.items() if code != 0]
    if failed:
        raise SystemExit('smoke failed for {}'.format(failed))
    for b in BRANCHES:
        shutil.rmtree('logs/smoke_{}'.format(b), ignore_errors=True)
    results.clear()
    print('\nsmoke ok')

In [ ]:
codes = run_pinned(
    [(b, 'apps/cifar100_{}.yml'.format(b)) for b in BRANCHES])
print(codes)

## Results, against A

A is the number to beat, not C or D. Improving on an ablation of your own
method is not improving on the paper.

One seed, and sigma has not been measured. Three of the four gaps in the
finished table sit between 0.25 and 0.44 points, which is the range where
a single run cannot tell a result from noise.

In [ ]:
# the published run, for reference
A_KL = {0.25: 70.10, 0.30: 70.80, 0.35: 71.50, 0.40: 72.20, 0.45: 72.80,
        0.50: 73.20, 0.55: 73.40, 0.60: 73.80, 0.65: 73.90, 0.70: 74.30,
        0.75: 74.60, 0.80: 74.80, 0.85: 75.10, 0.90: 75.10, 0.95: 75.40,
        1.00: 75.30}

widths = sorted({w for table in results.values() for w in table})
header = '{:>7}{:>9}'.format('width', 'A')
for branch in BRANCHES:
    header += '{:>11}{:>8}'.format(branch[:10], 'vs A')
print(header)

for width in widths:
    row = '{:>7.2f}{:>9.2f}'.format(width, A_KL.get(width, float('nan')))
    for branch in BRANCHES:
        entry = results.get(branch, {}).get(width)
        if entry is None:
            row += '{:>11}{:>8}'.format('-', '-')
            continue
        accuracy = 100.0 * (1.0 - entry[1])
        row += '{:>11.2f}{:>+8.2f}'.format(
            accuracy, accuracy - A_KL.get(width, accuracy))
    print(row)

print()
reference = sum(A_KL.values()) / len(A_KL)
print('{:22} mean {:.2f}   worst {:.2f}'.format(
    'A (reference)', reference, min(A_KL.values())))
for branch in BRANCHES:
    table = results.get(branch, {})
    if not table:
        continue
    accuracies = [100.0 * (1.0 - v[1]) for v in table.values()]
    mean = sum(accuracies) / len(accuracies)
    print('{:22} mean {:.2f}   worst {:.2f}   vs A {:+.2f}'.format(
        branch, mean, min(accuracies), mean - reference))

out = os.path.join(WORK, 'logs')
for branch in BRANCHES:
    log_dir = 'logs/cifar100_{}'.format(branch)
    if os.path.isdir(log_dir):
        shutil.copytree(log_dir, os.path.join(out, 'cifar100_' + branch),
                        dirs_exist_ok=True)
print('\ncheckpoints copied to', out)